In [1]:

import os
import re
import glob
from collections import Counter

import numpy as np
import pandas as pd
from fugashi import Tagger

tagger = Tagger()

FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)
    hapax_ratio = sum(1 for v in freq.values() if v == 1) / word_count

    freq_counter = Counter(freq.values())
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())
    yules_k = 10000 * (m2 - word_count) / (word_count ** 2)

    return hapax_ratio, yules_k


def extract_features(text):
    """累積文章全体から既存と同じ種類の特徴量を計算する"""

    comma_count = text.count("、")
    period_count = text.count("。")
    exclamation_count = text.count("！")
    question_count = text.count("？")

    # 文末記号を保持して文分割
    sentences = [
        s for s in re.findall(r"[^。！？]*[。！？]|[^。！？]+$", text)
        if s.strip()
    ]
    sentence_lengths = [len(s.strip()) for s in sentences if s.strip()]

    total_chars = len(text)
    hiragana_count = len(re.findall(r"[ぁ-ん]", text))
    katakana_count = len(re.findall(r"[ァ-ン]", text))
    kanji_count = len(re.findall(r"[一-龥]", text))
    alnum_count = len(re.findall(r"[a-zA-Z0-9]", text))

    pos_counter = Counter()
    particle_counter = Counter()
    all_words, function_words, content_words = [], [], []

    for word in tagger(text):
        pos = word.feature.pos1
        surface = word.surface

        if pos == "補助記号":
            continue

        if pos == "助詞":
            particle_counter[surface] += 1

        all_words.append(surface)
        pos_counter[pos] += 1

        if pos in FUNCTION_POS:
            function_words.append(surface)
        elif pos in CONTENT_POS:
            content_words.append(surface)

    word_count = len(all_words)
    function_count = len(function_words)
    content_count = len(content_words)

    f_per = function_count / word_count * 100 if word_count else 0
    c_per = content_count / word_count * 100 if word_count else 0

    hapax_all, yule_all = hapax_yule(all_words, word_count)
    hapax_func, yule_func = hapax_yule(function_words, function_count)
    hapax_cont, yule_cont = hapax_yule(content_words, content_count)

    total_particle = sum(particle_counter.values())

    features = {
        "総単語数": word_count,
        "機能語率": f_per,
        "内容語率": c_per,
        "異なり語数": len(set(all_words)),
        "TTR": len(set(all_words)) / word_count if word_count else 0,
        "ひらがな率": hiragana_count / total_chars if total_chars else 0,
        "カタカナ率": katakana_count / total_chars if total_chars else 0,
        "漢字率": kanji_count / total_chars if total_chars else 0,
        "英数字率": alnum_count / total_chars if total_chars else 0,
        "文長標準偏差": np.std(sentence_lengths) if sentence_lengths else 0,
        "最大文長": max(sentence_lengths) if sentence_lengths else 0,
        "Hapax率_all": hapax_all,
        "Yules_K_all": yule_all,
        "Hapax率_func": hapax_func,
        "Yules_K_func": yule_func,
        "Hapax率_cont": hapax_cont,
        "Yules_K_cont": yule_cont,
        "平均文長": np.mean(sentence_lengths) if sentence_lengths else 0,
        "名詞割合": pos_counter["名詞"] / word_count * 100 if word_count else 0,
        "動詞割合": pos_counter["動詞"] / word_count * 100 if word_count else 0,
        "形容詞割合": pos_counter["形容詞"] / word_count * 100 if word_count else 0,
        "形状詞割合": pos_counter["形状詞"] / word_count * 100 if word_count else 0,
        "助詞割合": pos_counter["助詞"] / word_count * 100 if word_count else 0,
        "助動詞割合": pos_counter["助動詞"] / word_count * 100 if word_count else 0,
        "副詞割合": pos_counter["副詞"] / word_count * 100 if word_count else 0,
        "連体詞割合": pos_counter["連体詞"] / word_count * 100 if word_count else 0,
        "接続詞割合": pos_counter["接続詞"] / word_count * 100 if word_count else 0,
        "感動詞割合": pos_counter["感動詞"] / word_count * 100 if word_count else 0,
        "読点頻度": comma_count / word_count if word_count else 0,
        "句点頻度": period_count / word_count if word_count else 0,
        "感嘆符数": exclamation_count,
        "疑問符数": question_count,
    }

    # 既存CSVにある助詞列をすべて用意
    particle_names = [
        "と", "は", "の", "で", "て", "に", "や", "を", "ば", "が",
        "も", "つつ", "から", "など", "へ", "まで", "か", "たり",
        "だけ", "ながら", "より"
    ]

    for p in particle_names:
        features[f"助詞_{p}"] = (
            particle_counter[p] / total_particle if total_particle else 0
        )

    return features


# 対象フォルダとラベル
folders = [
    ("../../test_Human_text", "Human"),
    ("../../test_AI_text", "AI"),
]

rows = []

for folder, label in folders:
    filepaths = sorted(glob.glob(os.path.join(folder, "*.txt")))

    if not filepaths:
        print(f"警告: {folder} にtxtファイルが見つかりません")

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # 元の文末記号を含めて文単位に分割
        sentences = [
            s.strip()
            for s in re.findall(r"[^。！？]*[。！？]|[^。！？]+$", text)
            if s.strip()
        ]

        for sent_order in range(1, len(sentences) + 1):
            cumulative_text = "".join(sentences[:sent_order])

            row = {
                "doc_id": os.path.basename(filepath),
                "label": label,
                "sent_order": sent_order,
                **extract_features(cumulative_text),
            }
            rows.append(row)

df_cumulative = pd.DataFrame(rows)

output_path = os.path.join(
    os.getcwd(), "test_cumulative_stylometry_features.csv"
)
df_cumulative.to_csv(output_path, index=False, encoding="utf-8-sig")

print("累積特徴量の作成完了")
print("文書・累積入力の組数:", len(df_cumulative))
print("保存先:", output_path)
display(df_cumulative.head())


累積特徴量の作成完了
文書・累積入力の組数: 206
保存先: /workspace/研究/one-sentence/AI確信度/test_cumulative_stylometry_features.csv


,doc_id,label,sent_order,総単語数,機能語率,内容語率,異なり語数,TTR,ひらがな率,カタカナ率,...,助詞_つつ,助詞_から,助詞_など,助詞_へ,助詞_まで,助詞_か,助詞_たり,助詞_だけ,助詞_ながら,助詞_より
0,test_Human_text_01.txt,Human,1,16,43.750000,56.250000,16,1.000000,0.411765,0.264706,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,test_Human_text_01.txt,Human,2,49,44.897959,53.061224,38,0.775510,0.534884,0.139535,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,test_Human_text_01.txt,Human,3,85,42.352941,49.411765,52,0.611765,0.520833,0.125000,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,test_Human_text_01.txt,Human,4,115,45.217391,48.695652,62,0.539130,0.470874,0.194175,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,test_Human_text_01.txt,Human,5,155,45.161290,49.032258,79,0.509677,0.494465,0.147601,...,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
import os
import pandas as pd

current_dir = os.getcwd()
print("作業フォルダ:", current_dir)
csv_path = os.path.join(
    current_dir,
    "test_cumulative_stylometry_features.csv"
)

df_cum = pd.read_csv(csv_path)

print("データサイズ:", df_cum.shape)
print("\n列名:")
print(df_cum.columns.tolist())

print("\n先頭5行:")
display(df_cum.head())

作業フォルダ: /workspace/研究/one-sentence/AI確信度
データサイズ: (206, 56)

列名:
['doc_id', 'label', 'sent_order', '総単語数', '機能語率', '内容語率', '異なり語数', 'TTR', 'ひらがな率', 'カタカナ率', '漢字率', '英数字率', '文長標準偏差', '最大文長', 'Hapax率_all', 'Yules_K_all', 'Hapax率_func', 'Yules_K_func', 'Hapax率_cont', 'Yules_K_cont', '平均文長', '名詞割合', '動詞割合', '形容詞割合', '形状詞割合', '助詞割合', '助動詞割合', '副詞割合', '連体詞割合', '接続詞割合', '感動詞割合', '読点頻度', '句点頻度', '感嘆符数', '疑問符数', '助詞_と', '助詞_は', '助詞_の', '助詞_で', '助詞_て', '助詞_に', '助詞_や', '助詞_を', '助詞_ば', '助詞_が', '助詞_も', '助詞_つつ', '助詞_から', '助詞_など', '助詞_へ', '助詞_まで', '助詞_か', '助詞_たり', '助詞_だけ', '助詞_ながら', '助詞_より']

先頭5行:


,doc_id,label,sent_order,総単語数,機能語率,内容語率,異なり語数,TTR,ひらがな率,カタカナ率,...,助詞_つつ,助詞_から,助詞_など,助詞_へ,助詞_まで,助詞_か,助詞_たり,助詞_だけ,助詞_ながら,助詞_より
0,test_Human_text_01.txt,Human,1,16,43.750000,56.250000,16,1.000000,0.411765,0.264706,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,test_Human_text_01.txt,Human,2,49,44.897959,53.061224,38,0.775510,0.534884,0.139535,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,test_Human_text_01.txt,Human,3,85,42.352941,49.411765,52,0.611765,0.520833,0.125000,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,test_Human_text_01.txt,Human,4,115,45.217391,48.695652,62,0.539130,0.470874,0.194175,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,test_Human_text_01.txt,Human,5,155,45.161290,49.032258,79,0.509677,0.494465,0.147601,...,0.02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:

import os
import pandas as pd

# 入力ファイル
proba_path = os.path.join(
    current_dir,
    "rf_predict_proba_split_All_results.csv"
)

features_path = os.path.join(
    current_dir,
    "test_cumulative_stylometry_features.csv"
)

# 読み込み
df_proba = pd.read_csv(proba_path)
df_features = pd.read_csv(features_path)

# 結合
df_merged = pd.merge(
    df_proba,
    df_features,
    on=["doc_id", "label", "sent_order"],
    how="inner",
    validate="one_to_one"
)

# 保存先
output_path = os.path.join(
    current_dir,
    "RF_predict_proba_cumulative_features_merged.csv"
)

df_merged.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# 結果確認
print("確信度CSVの行数:", len(df_proba))
print("特徴量CSVの行数:", len(df_features))
print("結合後の行数:", len(df_merged))
print("保存先:", output_path)

print("\n結合結果（先頭10行）:")
display(
    df_merged[
        ["doc_id", "label", "sent_order", "AI確率"]
    ].head(10)
)


確信度CSVの行数: 206
特徴量CSVの行数: 206
結合後の行数: 206
保存先: /workspace/研究/one-sentence/AI確信度/RF_predict_proba_cumulative_features_merged.csv

結合結果（先頭10行）:


,doc_id,label,sent_order,AI確率
0,test_Human_text_01.txt,Human,1,0.280000
1,test_Human_text_01.txt,Human,2,0.330000
2,test_Human_text_01.txt,Human,3,0.336667
3,test_Human_text_01.txt,Human,4,0.327500
4,test_Human_text_01.txt,Human,5,0.306000
5,test_Human_text_01.txt,Human,6,0.308333
6,test_Human_text_01.txt,Human,7,0.335714
7,test_Human_text_01.txt,Human,8,0.378750
8,test_Human_text_01.txt,Human,9,0.380000
9,test_Human_text_02.txt,Human,1,0.220000
